In [ ]:
import pandas as pd
pd.set_option('display.max_columns', None)
from google.colab import drive
drive.mount('/content/drive')

In [ ]:
#!/usr/bin/env python3
"""
diagnose_coverage.py  —  Google Colab
=====================================
Pourquoi seulement ~30 % ? Ce script répond, SANS rien télécharger en masse.

Il vérifie, dans l'ordre :
  1. IDENTIFIANTS DISPONIBLES — le fichier d'entrée contient-il réellement
     oa_doi / oa_pmcid / oa_pmid / upw_oa_url ? Un DOI manquant = Unpaywall
     jamais appelé = "no_oa_url" trompeur.
  2. UNPAYWALL FONCTIONNE-T-IL ? — test live sur un échantillon de DOI, avec
     le détail : is_oa, présence d'une URL PDF, d'une landing page.
  3. PMID -> PMCID — combien de lignes sans PMCID pourraient en obtenir un via
     le convertisseur NCBI (échantillon).
  4. EUROPE PMC PAR PMID — combien de lignes sans DOI ni PMCID sont trouvables
     par PMID.
  5. CHAÎNES D'ÉCHEC du manifeste, décodées en causes lisibles.

SETUP
  !pip install requests pandas pyarrow openpyxl -q
"""

import json, random, re, sys, time
from collections import Counter
from pathlib import Path
import pandas as pd
import requests

# =========================================================================== #
# CONFIG                                                                        #
# =========================================================================== #
IN_FILE   = "/content/drive/MyDrive/COGNOSX/ticks/preprocessing_data/merged_human_class2.xlsx"
INCLUDE_COL, INCLUDE_VAL = "last_final_include", "include"
MANIFEST  = "/content/drive/MyDrive/COGNOSX/ticks/fulltext3/fulltext_manifest.parquet"
MAILTO    = "cognosxx@gmail.com"
SAMPLE    = 60          # taille des échantillons testés en live

SESS = requests.Session()
SESS.headers.update({"User-Agent": f"tick-diag/1.0 (mailto:{MAILTO})"})


def S(x):
    return "" if x is None or (isinstance(x, float) and pd.isna(x)) or x is pd.NA else str(x)
def norm_doi(x):
    return S(x).lower().replace("https://doi.org/", "").replace("doi.org/", "").strip()
def clean_pmid(x):
    v = re.sub(r"\D", "", S(x))
    return v if v else ""


# =========================================================================== #
# 1. quels identifiants avons-nous réellement ?                               #
# =========================================================================== #
def check_ids(df):
    print("=" * 68)
    print("1. IDENTIFIANTS DISPONIBLES")
    print("=" * 68)
    print(f"lignes incluses : {len(df):,}\n")
    wanted = ["oa_id", "oa_doi", "oa_pmcid", "oa_pmid", "upw_oa_url",
              "oa_is_oa", "oa_oa_status"]
    for c in wanted:
        if c not in df.columns:
            print(f"  {c:<16} COLONNE ABSENTE   <-- ne peut pas être utilisée")
            continue
        n = df[c].map(lambda v: bool(S(v).strip())).sum()
        print(f"  {c:<16} {n:>7,} / {len(df):,}  ({n/len(df):6.1%})")

    have_doi   = df["oa_doi"].map(lambda v: bool(norm_doi(v))) if "oa_doi" in df else pd.Series(False, index=df.index)
    have_pmcid = df["oa_pmcid"].map(lambda v: bool(S(v).strip())) if "oa_pmcid" in df else pd.Series(False, index=df.index)
    have_pmid  = df["oa_pmid"].map(lambda v: bool(S(v).strip())) if "oa_pmid" in df else pd.Series(False, index=df.index)

    print(f"\n  ni DOI ni PMCID ni PMID : {int((~have_doi & ~have_pmcid & ~have_pmid).sum()):,}"
          f"   <-- irrécupérables automatiquement")
    print(f"  PMID mais pas de PMCID  : {int((have_pmid & ~have_pmcid).sum()):,}"
          f"   <-- récupérables via le convertisseur NCBI")
    print(f"  PMID mais ni DOI ni PMCID : {int((have_pmid & ~have_pmcid & ~have_doi).sum()):,}"
          f" <-- actuellement JAMAIS tentés (try_epmc ne cherche que par DOI)")
    print(f"  DOI présent (Unpaywall possible) : {int(have_doi.sum()):,}")
    return have_doi, have_pmcid, have_pmid


# =========================================================================== #
# 2. Unpaywall fonctionne-t-il vraiment ?                                     #
# =========================================================================== #
def test_unpaywall(df, have_doi):
    print("\n" + "=" * 68)
    print("2. TEST LIVE UNPAYWALL")
    print("=" * 68)
    dois = [norm_doi(d) for d in df.loc[have_doi, "oa_doi"]]
    if not dois:
        print("  aucun DOI -> Unpaywall inutilisable"); return
    samp = random.Random(42).sample(dois, min(SAMPLE, len(dois)))
    c = Counter(); examples = []
    for d in samp:
        try:
            r = SESS.get(f"https://api.unpaywall.org/v2/{d}",
                         params={"email": MAILTO}, timeout=25)
            if r.status_code != 200:
                c[f"http_{r.status_code}"] += 1; continue
            j = r.json()
            is_oa = bool(j.get("is_oa"))
            best = j.get("best_oa_location") or {}
            pdf = best.get("url_for_pdf"); land = best.get("url")
            if not is_oa:
                c["closed (is_oa=false)"] += 1
            elif pdf:
                c["OA + url_for_pdf"] += 1
                if len(examples) < 3: examples.append((d, pdf))
            elif land:
                c["OA mais landing seulement"] += 1
                if len(examples) < 3: examples.append((d, land))
            else:
                c["OA mais AUCUNE url"] += 1
        except requests.RequestException as e:
            c[f"erreur reseau"] += 1
        time.sleep(0.15)
    print(f"  échantillon de {len(samp)} DOI :")
    for k, v in c.most_common():
        print(f"    {k:<28}{v:>5}  ({v/len(samp):5.1%})")
    if examples:
        print("\n  exemples d'URL retournées :")
        for d, u in examples:
            print(f"    {d}\n      -> {u[:95]}")
    oa_rate = (c["OA + url_for_pdf"] + c["OA mais landing seulement"]) / max(len(samp), 1)
    print(f"\n  => Unpaywall trouve un accès ouvert pour {oa_rate:.1%} des DOI testés.")
    if oa_rate > 0.1:
        print("     Unpaywall FONCTIONNE. Si vos lignes échouent en 'no_oa_url',")
        print("     c'est que le DOI est absent, pas qu'Unpaywall est mal réglé.")


# =========================================================================== #
# 3. PMID -> PMCID via le convertisseur NCBI                                   #
# =========================================================================== #
def test_pmid_to_pmcid(df, have_pmcid, have_pmid):
    print("\n" + "=" * 68)
    print("3. PMID -> PMCID (convertisseur NCBI)")
    print("=" * 68)
    sub = df[have_pmid & ~have_pmcid]
    if not len(sub):
        print("  aucune ligne avec PMID sans PMCID"); return
    pmids = [S(p).strip() for p in sub["oa_pmid"] if S(p).strip().isdigit()]
    samp = random.Random(42).sample(pmids, min(SAMPLE * 3, len(pmids)))
    found = 0
    try:
        r = SESS.get("https://pmc.ncbi.nlm.nih.gov/tools/idconv/api/v1/articles/",
                     params={"ids": ",".join(samp), "format": "json",
                             "email": MAILTO, "idtype": "pmid"}, timeout=60)
        if r.status_code == 200:
            for rec in r.json().get("records", []):
                if rec.get("pmcid"): found += 1
        else:
            print(f"  convertisseur: HTTP {r.status_code}"); return
    except Exception as e:
        print(f"  erreur: {e}"); return
    rate = found / max(len(samp), 1)
    print(f"  {found}/{len(samp)} PMID ont un PMCID  ({rate:.1%})")
    print(f"  => extrapolé sur {len(pmids):,} lignes : ~{int(rate*len(pmids)):,} "
          f"PMCID récupérables")
    print("     Chacun bascule vers Europe PMC = JATS structuré (la meilleure source).")


# =========================================================================== #
# 4. Europe PMC par PMID (pour les lignes sans DOI ni PMCID)                   #
# =========================================================================== #
def test_epmc_by_pmid(df, have_doi, have_pmcid, have_pmid):
    print("\n" + "=" * 68)
    print("4. EUROPE PMC PAR PMID (lignes sans DOI ni PMCID)")
    print("=" * 68)
    sub = df[have_pmid & ~have_pmcid & ~have_doi]
    if not len(sub):
        print("  aucune ligne concernée"); return
    pmids = [S(p).strip() for p in sub["oa_pmid"] if S(p).strip().isdigit()]
    samp = random.Random(42).sample(pmids, min(30, len(pmids)))
    hit = 0
    for p in samp:
        try:
            r = SESS.get("https://www.ebi.ac.uk/europepmc/webservices/rest/search",
                         params={"query": f"EXT_ID:{p} AND SRC:MED",
                                 "format": "json", "resultType": "idlist"}, timeout=25)
            if r.status_code == 200:
                res = r.json().get("resultList", {}).get("result", [])
                if res and res[0].get("pmcid"): hit += 1
        except requests.RequestException:
            pass
        time.sleep(0.15)
    print(f"  {hit}/{len(samp)} trouvés avec un PMCID sur Europe PMC ({hit/max(len(samp),1):.1%})")
    print(f"  => ~{int(hit/max(len(samp),1)*len(pmids)):,} lignes actuellement perdues "
          f"seraient récupérables")


# =========================================================================== #
# 5. chaînes d'échec décodées                                                 #
# =========================================================================== #
CAUSE = {
    "no_pmcid": "pas de PMCID (ni via recherche DOI sur EPMC)",
    "no_doi": "pas de DOI dans le fichier d'entrée",
    "no_oa_url": "Unpaywall n'a retourné aucune URL (souvent: pas de DOI, ou accès fermé)",
    "web_failed": "URL trouvée mais téléchargement/parsing échoué",
    "grobid_off": "étage PDF sauté (GROBID désactivé)",
    "not_preprint": "pas un préprint bioRxiv/medRxiv (normal)",
    "no_arxiv": "pas un arXiv (normal)",
    "epmc_notfound": "EPMC: PMCID inconnu",
    "epmc_unparsable": "XML reçu mais corps trop court / illisible",
    "pmc_unparsable": "XML PMC reçu mais corps trop court / illisible",
}

def decode_failures():
    print("\n" + "=" * 68)
    print("5. CHAÎNES D'ÉCHEC")
    print("=" * 68)
    if not Path(MANIFEST).exists():
        print("  manifeste introuvable"); return
    man = pd.read_parquet(MANIFEST)
    fail = man[~man["ok"].fillna(False)]
    ok = man[man["ok"].fillna(False)]
    print(f"  manifeste : {len(man):,} lignes | ok {len(ok):,} | échecs {len(fail):,}")
    if len(ok):
        print("\n  sources réussies :")
        for k, v in ok["source"].value_counts().head(12).items():
            print(f"    {k:<32}{v:>7,}")
        if "has_targets" in ok:
            print(f"\n  avec sections cibles : {int(ok['has_targets'].sum()):,} "
                  f"({ok['has_targets'].mean():.1%} des réussites)")
    if not len(fail): return
    print(f"\n  top chaînes d'échec :")
    for k, v in fail["attempts"].value_counts().head(10).items():
        print(f"    {v:>7,}  {k}")
    # dernière étape = cause réelle
    last = fail["attempts"].astype(str).str.split(">").str[-1]
    print(f"\n  CAUSE FINALE (dernière étape tentée) :")
    for k, v in last.value_counts().head(10).items():
        print(f"    {v:>7,}  {k:<22} {CAUSE.get(k, '')}")


# =========================================================================== #
if __name__ == "__main__":
    p = str(IN_FILE).lower()
    df_all = (pd.read_excel(IN_FILE, dtype=str) if p.endswith((".xlsx", ".xls"))
              else pd.read_parquet(IN_FILE))
    df = (df_all[df_all[INCLUDE_COL].astype(str).str.strip().str.lower() == INCLUDE_VAL]
          if INCLUDE_COL in df_all.columns else df_all)

    have_doi, have_pmcid, have_pmid = check_ids(df)
    test_unpaywall(df, have_doi)
    test_pmid_to_pmcid(df, have_pmcid, have_pmid)
    test_epmc_by_pmid(df, have_doi, have_pmcid, have_pmid)
    decode_failures()

    print("\n" + "=" * 68)
    print("CORRECTIFS IMMÉDIATS")
    print("=" * 68)
    print("  a) discover_pdf_links : \"fulltext3\" -> \"fulltext\"  (remplacement")
    print("     accidentel lors du renommage du dossier ; casse la découverte PDF)")
    print("  b) ajouter la recherche EPMC par PMID dans try_epmc (voir patch)")
    print("  c) pré-remplir oa_pmcid via le convertisseur NCBI avant de relancer")
    print("  d) NCBI_KEY vide -> 3 req/s au lieu de 10 (gratuit à obtenir)")